### YOLOv11n Transfer Learning from Fashionpedia Dataset

In [11]:
import os
import tqdm
import numpy as np
from datasets import load_dataset
from ultralytics import YOLO

In [12]:
#Download Dataset
ds = load_dataset("detection-datasets/fashionpedia")

#### Convert COCO to YOLO

In [13]:
# Create directory structure
output_dir = "/home/tommytang111/Projects/Drone2/data/yolo_format"
os.makedirs(f"{output_dir}/images/train", exist_ok=True)
os.makedirs(f"{output_dir}/images/val", exist_ok=True)  
os.makedirs(f"{output_dir}/labels/train", exist_ok=True)
os.makedirs(f"{output_dir}/labels/val", exist_ok=True)

In [14]:
# Get class names
class_names = ds["train"].features["objects"].feature["category"].names
num_classes = len(class_names)
print(f"Found {num_classes} classes in Fashionpedia dataset")

Found 46 classes in Fashionpedia dataset


In [25]:
def coco_to_yolo_bbox(bbox, img_width, img_height):
    """Convert COCO format [x_min, y_min, width, height] to YOLO format [x_center, y_center, width, height] (normalized)"""
    x_min, y_min, width, height = bbox
    
    # Handle edge cases with invalid bounding boxes
    if width <= 0 or height <= 0:
        return None
        
    # Convert to YOLO format (normalized)
    x_center = (x_min + width / 2) / img_width
    y_center = (y_min + height / 2) / img_height
    width = width / img_width
    height = height / img_height
    
    # Ensure values are in valid range [0, 1]
    if not (0 <= x_center <= 1 and 0 <= y_center <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None
        
    return [x_center, y_center, width, height]

In [26]:
# Process each split
for split in ["train", "val"]:
    yolo_split = "train" if split == "train" else "val"
    print(f"Processing {split} split...")
    
    for i, item in enumerate(tqdm.tqdm(ds[split])):
        # Get image
        img = item["image"]
        img_width, img_height = img.size
        
        # Create unique filename based on index
        filename = f"{i:06d}"
        
        # Save image
        img_path = f"{output_dir}/images/{yolo_split}/{filename}.jpg"
        img.save(img_path)
        
        # Save YOLO label
        label_path = f"{output_dir}/labels/{yolo_split}/{filename}.txt"
        
        with open(label_path, "w") as f:
            # Process each object
            for j in range(len(item["objects"]["bbox"])):
                # Get class ID and bounding box
                class_id = item["objects"]["category"][j]
                bbox = item["objects"]["bbox"][j]
                
                # Convert to YOLO format
                yolo_bbox = coco_to_yolo_bbox(bbox, img_width, img_height)
                
                # Skip invalid bounding boxes
                if yolo_bbox is None:
                    continue
                
                # Write to file: class_id x_center y_center width height
                bbox_str = " ".join([f"{coord:.6f}" for coord in yolo_bbox])
                f.write(f"{class_id} {bbox_str}\n")

Processing train split...


100%|██████████| 45623/45623 [03:42<00:00, 205.43it/s]


Processing val split...


100%|██████████| 1158/1158 [00:06<00:00, 187.36it/s]


In [27]:
# Create data.yaml file
yaml_content = f"""
train: {output_dir}/images/train
val: {output_dir}/images/val

nc: {num_classes}
names: {list(class_names)}
"""

with open(f"{output_dir}/data.yaml", "w") as f:
    f.write(yaml_content)

print(f"Conversion complete. Dataset saved to {output_dir}")
print(f"Created data.yaml with {num_classes} classes")

Conversion complete. Dataset saved to /home/tommytang111/Projects/Drone2/data/yolo_format
Created data.yaml with 46 classes


In [28]:
# Examine dataset structure and verify conversion success
!find {output_dir} -type f | wc -l
print("Sample label file content:")
!head -n 3 {output_dir}/labels/train/000000.txt

# Check class distribution
import glob
import re

def count_classes(label_dir):
    class_counts = [0] * num_classes
    for label_file in glob.glob(f"{label_dir}/*.txt"):
        with open(label_file, 'r') as f:
            for line in f:
                class_id = int(line.split()[0])
                class_counts[class_id] += 1
    return class_counts

train_class_counts = count_classes(f"{output_dir}/labels/train")
val_class_counts = count_classes(f"{output_dir}/labels/val")

# Display top 10 classes
top_classes = sorted(range(len(train_class_counts)), 
                    key=lambda i: train_class_counts[i], 
                    reverse=True)[:10]

print("\nTop 10 classes by frequency:")
for i, class_id in enumerate(top_classes):
    print(f"{i+1}. {class_names[class_id]}: {train_class_counts[class_id]} train, {val_class_counts[class_id]} val")

93563
Sample label file content:
33 0.719941 0.447266 0.565982 0.343750
10 0.636364 0.600098 0.656891 0.649414

Top 10 classes by frequency:
1. sleeve: 45086 train, 1211 val
2. neckline: 33571 train, 894 val
3. pocket: 19116 train, 388 val
4. dress: 18478 train, 495 val
5. top, t-shirt, sweatshirt: 16083 train, 453 val
6. collar: 9978 train, 215 val
7. jacket: 7694 train, 177 val
8. pants: 7266 train, 218 val
9. zipper: 6520 train, 152 val
10. shirt, blouse: 6056 train, 102 val


#### Training

In [15]:
#Load Model
model = YOLO('yolo11x')

100%|██████████| 109M/109M [00:01<00:00, 58.9MB/s] 


In [ ]:
# Train YOLOv11n on Fashionpedia
results = model.train(
    data=f'{output_dir}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,  # Reduced batch size to start with, increase if GPU memory allows
    device=0,
    pretrained=True,
    patience=10,
    name='yolov11x-fashionpedia'
)

print(f"Training complete. Best model saved at: {results.best}")

New https://pypi.org/project/ultralytics/8.3.133 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.96 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3070, 8192MiB)
engine/trainer: task=detect, mode=train, model=yolo11x.pt, data=/home/tommytang111/Projects/Drone2/data/yolo_format/data.yaml, epochs=100, time=None, patience=10, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=8, project=None, name=yolov11x-fashionpedia, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes

train: Scanning /home/tommytang111/Projects/Drone2/data/yolo_format/labels/train.cache... 45623 images, 206 backgrounds, 0 corrupt: 100%|██████████| 45623/45623 [00:00<?, ?it/s]


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Scanning /home/tommytang111/Projects/Drone2/data/yolo_format/labels/val.cache... 1158 images, 14 backgrounds, 0 corrupt: 100%|██████████| 1158/1158 [00:00<?, ?it/s]


Plotting labels to runs/detect/yolov11x-fashionpedia/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: SGD(lr=0.01, momentum=0.9) with parameter groups 167 weight(decay=0.0), 174 weight(decay=0.0005), 173 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to runs/detect/yolov11x-fashionpedia
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      1/100      16.1G      2.726       4.67      3.274        188        640:   0%|          | 4/2852 [02:51<31:10:21, 39.40s/it]